# Synthetic vector-borne disease model

Humans (SEIRS, four age bands) and mosquitoes (SEI) coupled by Ross-Macdonald biting. No real
data: every parameter is illustrative. The model is built in `vbd_model.model`, whose module
docstring gives the force-of-infection formulas.

Age heterogeneity enters in two places:

- **bite rate**: `rel_bite_<age>` is each band's relative attractiveness to mosquitoes. Bites are
  shared out in proportion to `rel_bite × population`, so it affects both who gets infected and
  who infects mosquitoes;
- **susceptibility**: `rel_susc_<age>` scales the chance an infectious bite infects.

In [ ]:
import numpy as np
import pandas as pd

pd.options.plotting.backend = "plotly"

from summer4 import Compartments, Epoch, FlowMass, OutputSet

from vbd_model.model import DEFAULT_PARAMS, HUMANS, MOSQUITOES_PER_HUMAN, age, build_model, hstate, mstate, r0, species

In [ ]:
params = dict(DEFAULT_PARAMS)
pd.Series(params, name="value").to_frame()

In [ ]:
model = build_model()
compiled = model.compile()
compiled.pmap.labels()

R0 from the next-generation matrix (`r0`), with and without the age heterogeneity:

In [ ]:
flat = {**params, **{f"rel_bite_{a}": 1.0 for a in age.traits}, **{f"rel_susc_{a}": 1.0 for a in age.traits}}
pd.Series({"with age heterogeneity": r0(params), "no age effects (all multipliers 1)": r0(flat)}, name="R0")

## Run for ten years

In [ ]:
outputs = OutputSet()
outputs["incidence_by_age"] = FlowMass("human_infection").sum_by(age, side="source")
outputs["population_by_age"] = Compartments(where=species["human"]).sum_by(age)
outputs["human_prevalence"] = Compartments(where=hstate["I"]).sum_by() / HUMANS
outputs["mosquito_prevalence"] = Compartments(where=mstate["I"]).sum_by() / (HUMANS * MOSQUITOES_PER_HUMAN)
outputs["immune_by_age"] = Compartments(where=hstate["R"]).sum_by(age) / outputs.ref("population_by_age")

epoch = Epoch(pd.Timestamp("2025-01-01"))
run_kwargs = dict(t0=0.0, t1=10 * 365.0, dt=1.0, epoch=epoch, solver="tsit5")
result = compiled.run(params, save=outputs.plan(), **run_kwargs)
assert result.solver.ok
named = outputs.evaluate(result, params)

In [ ]:
prevalence = pd.concat(
    [named["human_prevalence"].to_pandas()["value"].rename("humans infectious"),
     named["mosquito_prevalence"].to_pandas()["value"].rename("mosquitoes infectious")],
    axis=1,
)
prevalence.plot(title="Infectious prevalence", labels={"value": "proportion", "time": ""})

Annual infections per 1,000 people, by age band:

In [ ]:
annual = named["incidence_by_age"].integrate_intervals().resample("YE", how="sum").to_pandas()
pop = named["population_by_age"].to_pandas().iloc[0]
annual_rate = 1000 * annual / pop.values
annual_rate.index = annual_rate.index.year
annual_rate.round(0)

With the defaults, the two gradients pull in opposite directions: children are bitten least
(`rel_bite_0 = 0.5`) but are the most susceptible (`rel_susc_0 = 1.0`), so the product
`rel_bite × rel_susc` is 0.50, 0.72, 0.70 and 0.66 across the bands. Per-capita incidence ends up
nearly flat by age: in the endemic years (2030 onwards) about 410 per 1,000 per year in under-5s
against 360 in the 40+ band. Under-5s are slightly higher because they are born susceptible.
The heterogeneity still matters for transmission: bites concentrated on adults, who are less
susceptible, lower R0 from 2.37 to 1.97.

In [ ]:
annual_rate.plot(kind="bar", barmode="group", title="Infections per 1,000 per year, by age",
                 labels={"value": "per 1,000", "index": "year"})

In [ ]:
named["immune_by_age"].to_pandas().plot(title="Proportion immune (R), by age",
                                        labels={"value": "proportion", "time": ""})

## Scenario: halve the bite rate

A crude stand-in for bednets or larviciding. `bite_rate` is a parameter, so this reuses the
compiled model.

In [ ]:
scenario = {**params, "bite_rate": params["bite_rate"] / 2}
print(f"R0: baseline {r0(params):.2f}, halved bite rate {r0(scenario):.2f}")
named_scenario = outputs.evaluate(compiled.run(scenario, save=outputs.plan(), **run_kwargs), scenario)
pd.concat(
    [named["human_prevalence"].to_pandas()["value"].rename("baseline"),
     named_scenario["human_prevalence"].to_pandas()["value"].rename("bite rate halved")],
    axis=1,
).plot(title="Human infectious prevalence", labels={"value": "proportion", "time": ""})

Halving the bite rate halves R0 (to 0.98, since R0 is proportional to the bite rate), and
infection dies out instead of becoming endemic.